In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — 03: Backfill Historical Data
# MAGIC
# MAGIC Backfills the Bronze table with historical WattTime CAISO_NORTH data.
# MAGIC Run this once after initial setup to populate the full date range.
# MAGIC Bronze table must exist before running — see `00_setup/01_create_schemas_and_tables`.

# COMMAND ----------

from databricks.sdk import WorkspaceClient
import requests
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

CATALOG = "bootcamp_students"
BRONZE_TABLE = f"{CATALOG}.bronze.watttime_raw"
REGION = "CAISO_NORTH"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Authenticate with WattTime

# COMMAND ----------

username = w.secrets.get_secret(scope="arc-lab", key="watttime-username").value
password = w.secrets.get_secret(scope="arc-lab", key="watttime-password").value

token_resp = requests.get(
    "https://api.watttime.org/login",
    auth=(username, password)
)
token_resp.raise_for_status()
TOKEN = token_resp.json()["token"]
print("Authentication successful.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Define Backfill Window

# COMMAND ----------

# Adjust start/end to cover desired historical range
BACKFILL_START = "2025-01-01T00:00+00:00"
BACKFILL_END   = "2025-12-31T23:59+00:00"

print(f"Backfill window: {BACKFILL_START} → {BACKFILL_END}")
print(f"Target region:   {REGION}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Fetch Historical Data from WattTime

# COMMAND ----------

headers = {"Authorization": f"Bearer {TOKEN}"}

params = {
    "region":    REGION,
    "starttime": BACKFILL_START,
    "endtime":   BACKFILL_END,
    "signal_type": "co2_moer"
}

response = requests.get(
    "https://api.watttime.org/v3/historical",
    headers=headers,
    params=params
)
response.raise_for_status()
data = response.json().get("data", [])
print(f"Records fetched: {len(data)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Write to Bronze Table (Append)

# COMMAND ----------

if data:
    rows = [
        (
            rec.get("point_time"),
            float(rec.get("value", 0)),
            rec.get("ba", REGION),
            rec.get("freq", "300"),
            datetime.now(timezone.utc).isoformat()
        )
        for rec in data
    ]

    schema = StructType([
        StructField("ts_utc",        StringType(),  True),
        StructField("value",         DoubleType(),  True),
        StructField("ba",            StringType(),  True),
        StructField("freq",          StringType(),  True),
        StructField("ingestion_ts",  StringType(),  True),
    ])

    df = spark.createDataFrame(rows, schema=schema)
    df.write.mode("append").saveAsTable(BRONZE_TABLE)
    print(f"✅ {len(rows)} rows appended to {BRONZE_TABLE}")
else:
    print("⚠️ No data returned. Check date range and WattTime token.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Verify

# COMMAND ----------

count = spark.sql(f"SELECT COUNT(*) as n FROM {BRONZE_TABLE}").collect()[0]["n"]
print(f"Total rows in Bronze after backfill: {count:,}")

---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File <command-7175539904832449>, line 38
     32 password = w.secrets.get_secret(scope="arc-lab", key="watttime-password").value
     34 token_resp = requests.get(
     35     "https://api.watttime.org/login",
     36     auth=(username, password)
     37 )
---> 38 token_resp.raise_for_status()
     39 TOKEN = token_resp.json()["token"]
     40 print("Authentication successful.")

File /databricks/python/lib/python3.12/site-packages/requests/models.py:1024, in Response.raise_for_status(self)
   1019     http_error_msg = (
   1020         f"{self.status_code} Server Error: {reason} for url: {self.url}"
   1021     )
   1023 if http_error_msg:
-> 1024     raise HTTPError(http_error_msg, response=self)

HTTPError: 403 Client Error: Forbidden for url: https://api.watttime.org/login